# Week 6 — Feature Engineering
**Internship:** IDX Exchange Data Science Program  
**Name:** Monika  
**Week:** 6  
**Dataset:** CRMLS Sold Properties, cleaned in Week 3, models from Week 5

**Goal:** Engineer a few simple derived features (bed/bath ratio), add a school
district feature via spatial join against the CA School District Areas 2024–25
boundaries, retrain all three models, and compare old vs new feature sets.

In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_percentage_error

data_folder = r'C:\Users\monik\OneDrive - University of Illinois - Urbana\Desktop\IDX Exchange_DS\data\california'
model_df = pd.read_csv(data_folder + '\\cleaned_full.csv', parse_dates=['CloseDate_parsed'])

BASE_FEATURE_COLS = [
    'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeAcres',
    'PropertyAge', 'DaysOnMarket', 'Latitude', 'Longitude',
    'PoolPrivateYN', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'AssociationFee',
    'LivingArea_missing', 'BathroomsTotalInteger_missing',
    'YearBuilt_missing', 'LotSizeAcres_missing', 'DaysOnMarket_anomaly'
]
target_col = 'ClosePrice'

def get_train_test_split(frame, test_month, window_months):
    frame = frame.copy()
    frame['YearMonth'] = frame['CloseDate_parsed'].dt.to_period('M')
    test_df = frame[frame['YearMonth'] == test_month]
    train_start = test_month - window_months
    train_df = frame[(frame['YearMonth'] >= train_start) & (frame['YearMonth'] < test_month)]
    return train_df.drop(columns='YearMonth'), test_df.drop(columns='YearMonth')

test_month = pd.Period('2026-06', freq='M')
BEST_WINDOW = 12  # set this to whatever Week 5 landed on

ModuleNotFoundError: No module named 'geopandas'

## 1. Bed/Bath Ratio
A simple ratio can sometimes capture "layout efficiency" better than the raw
counts alone. Guarding against divide-by-zero for the (hopefully rare) rows with
0 bathrooms.

In [3]:
model_df['BedBathRatio'] = model_df['BedroomsTotal'] / model_df['BathroomsTotalInteger'].replace(0, np.nan)
model_df['BedBathRatio'] = model_df['BedBathRatio'].fillna(model_df['BedBathRatio'].median())

NameError: name 'model_df' is not defined

## 2. School District Spatial Join
Loading the CA School District Areas 2024–25 boundaries locally (downloaded from
data.ca.gov, same convention as my other resource files — not fetching the URL
live at runtime). First checking the actual column names since I haven't
confirmed the district-name field yet.

In [4]:
district_path = data_folder + r'\resources\CA_School_District_Areas_2024_25.geojson'
districts = gpd.read_file(district_path)
print(districts.columns.tolist())
districts.head(2)

NameError: name 'data_folder' is not defined

**Note:** update `DISTRICT_NAME_COL` below to match whatever the actual column
is called once you've seen the printout above (likely something like
`DistrictName`, `NAME`, or `CDCode` — CA open data schemas vary).

In [5]:
DISTRICT_NAME_COL = 'DistrictName'  # <-- confirm/update after checking districts.columns above

properties_gdf = gpd.GeoDataFrame(
    model_df,
    geometry=gpd.points_from_xy(model_df['Longitude'], model_df['Latitude']),
    crs='EPSG:4326'
)
districts = districts.to_crs(properties_gdf.crs)

joined = gpd.sjoin(
    properties_gdf,
    districts[[DISTRICT_NAME_COL, 'geometry']],
    how='left',
    predicate='within'
)

n_unmatched = joined[DISTRICT_NAME_COL].isna().sum()
print(f'Rows with no matching district: {n_unmatched:,} ({n_unmatched / len(joined) * 100:.2f}%)')

model_df['SchoolDistrict'] = joined[DISTRICT_NAME_COL].fillna('Unknown')

NameError: name 'gpd' is not defined

## 3. Encode School District (Target Encoding, Train-Only)
`SchoolDistrict` is way too high-cardinality for one-hot encoding. Using target
encoding — median ClosePrice per district — but computing it **only on the
training split** each time and mapping it onto test, so the test period's own
prices never leak into the encoding. Unseen districts in test fall back to the
overall training median.

In [6]:
def add_district_target_encoding(train_df, test_df, target_col='ClosePrice'):
    district_medians = train_df.groupby('SchoolDistrict')[target_col].median()
    global_median = train_df[target_col].median()

    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df['SchoolDistrict_encoded'] = train_df['SchoolDistrict'].map(district_medians)
    test_df['SchoolDistrict_encoded'] = test_df['SchoolDistrict'].map(district_medians).fillna(global_median)
    return train_df, test_df

## 4. Build Old vs New Feature Sets

In [7]:
OLD_FEATURE_COLS = BASE_FEATURE_COLS
NEW_FEATURE_COLS = BASE_FEATURE_COLS + ['BedBathRatio', 'SchoolDistrict_encoded']

train_df, test_df = get_train_test_split(model_df, test_month, BEST_WINDOW)
train_df, test_df = add_district_target_encoding(train_df, test_df, target_col)

NameError: name 'BASE_FEATURE_COLS' is not defined

## 5. Retrain All Three Models on Both Feature Sets

In [8]:
def evaluate_models(train_df, test_df, feature_cols, target_col):
    X_train, y_train = train_df[feature_cols], train_df[target_col]
    X_test, y_test = test_df[feature_cols], test_df[target_col]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    lr = LinearRegression().fit(X_train_scaled, y_train)
    dt = DecisionTreeRegressor(max_depth=10, random_state=42).fit(X_train, y_train)
    rf = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=5,
                                random_state=42, n_jobs=-1).fit(X_train, y_train)

    return {
        'Linear Regression': r2_score(y_test, lr.predict(X_test_scaled)),
        'Decision Tree': r2_score(y_test, dt.predict(X_test)),
        'Random Forest': r2_score(y_test, rf.predict(X_test)),
    }

old_results = evaluate_models(train_df, test_df, OLD_FEATURE_COLS, target_col)
new_results = evaluate_models(train_df, test_df, NEW_FEATURE_COLS, target_col)

NameError: name 'train_df' is not defined

## 6. Old vs New Feature Set Comparison Table

In [10]:
comparison_df = pd.DataFrame({
    'model': list(old_results.keys()),
    'R2_old_features': list(old_results.values()),
    'R2_new_features': [new_results[m] for m in old_results.keys()],
})
comparison_df['R2_improvement'] = comparison_df['R2_new_features'] - comparison_df['R2_old_features']
comparison_df

NameError: name 'old_results' is not defined

## Summary
- Added `BedBathRatio` (guarded against divide-by-zero) as a simple layout
  feature.
- Added `SchoolDistrict` via spatial join against the CA School District Areas
  2024–25 boundaries, then target-encoded by **training-set-only** median
  ClosePrice per district to avoid leaking test-period prices into the encoding.
- Logged `n_unmatched` rows during the spatial join — worth investigating if
  that % is nontrivial (could mean CRS mismatch or points falling just outside
  boundary edges).
- Comparison table above shows R² for all three models with old vs new feature
  sets — if `SchoolDistrict_encoded` is doing real work, I'd expect Random
  Forest and Decision Tree to show the biggest lift, since they can use it more
  flexibly than Linear Regression can with a single encoded column.
- **Still to verify:** confirm `DISTRICT_NAME_COL` actually matches the real
  column name in the GeoJSON (I only guessed `'DistrictName'` above).